In [1]:
import os
import json

from pydantic import BaseModel
from openai import OpenAI

# 食材
class Ingredient(BaseModel):
    name: str
    amount: str
    kcal: int

# 食谱
class Recipe(BaseModel):
    ingredients: list[Ingredient]
    instructions: str

# 创建DashScope客户端(兼容OpenAI协议)
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 在system prompt中描述JSON格式
schema_desc = json.dumps(Recipe.model_json_schema(), ensure_ascii=False, indent=2)

# 通过response_format进行结构化输出
completion = client.chat.completions.create(
    model="qwen-turbo",
    messages=[
        {
            "role": "system",
            "content": f"你是一个食谱助手。请严格按照以下JSON Schema格式返回结果:\n{schema_desc}",
        },
        {"role": "user", "content": "请写一个苹果派的食谱"},
    ],
    response_format={"type": "json_object"},
)

apple_pie_recipe = Recipe.model_validate_json(completion.choices[0].message.content)

print(apple_pie_recipe)

#print(apple_pie_recipe.instructions)

#print(completion.choices[0].message.content)


ingredients=[Ingredient(name='苹果', amount='6个', kcal=120), Ingredient(name='面粉', amount='2杯', kcal=800), Ingredient(name='黄油', amount='1/2杯', kcal=400), Ingredient(name='糖', amount='1/2杯', kcal=350), Ingredient(name='肉桂粉', amount='1茶匙', kcal=10), Ingredient(name='柠檬汁', amount='1汤匙', kcal=5)] instructions='1. 将苹果削皮、切片并加入柠檬汁中。2. 在一个碗中，将面粉和黄油混合，直到形成颗粒状。3. 将糖和肉桂粉加入面团中，搅拌均匀。4. 将一半的面团放入烤盘中，铺上苹果片。5. 将剩余的面团放在苹果片上，用叉子在顶部戳几个孔。6. 在预热至180°C的烤箱中烤约45分钟，直到表面呈金黄色。7. 冷却后即可享用。'
